In [6]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scanpy as sc
import pandas as pd
import subprocess, os

In [ ]:
def run_canvas(dsetname, filename_parser):
   os.makedirs(f'_embeddings', exist_ok=True)

   subprocess.run(["python", "2.canvas_run.sh", dsetname])

   embedding = np.load(f'_data/{dsetname}/canvas/data/analysis/tile_embedding/embedding_mean.npy')
   print(embedding.shape)
   sample_names = np.load(f'_data/{dsetname}/canvas/data/analysis/tile_embedding/sample_name.npy')
   unique_samples = np.unique(sample_names)
   print(f'Loaded {len(sample_names)} samples, {len(unique_samples)} unique samples')
   obs = pd.DataFrame(sample_names, columns=['fullname'])
   obs.fullname = obs.fullname.str.replace('-', '.')
   obs['sid'] = obs.fullname.apply(lambda x: filename_parser(x)['sid'])
   obs['donor'] = obs.fullname.apply(lambda x: filename_parser(x)['donor'])
   obs['method_cluster'] = np.load(f'_data/{dsetname}/canvas/data/analysis/kmeans/10/clusters.npy')

   d = sc.AnnData(X=embedding,
                  obs=obs)
   sc.pp.neighbors(d)
   sc.tl.umap(d)
   sc.tl.leiden(d, resolution=1, key_added='leiden1')
   d.write(f'_embeddings/{dsetname}_canvas_noharm.h5ad')

# Run

In [ ]:
def alz_filename_parser(fname):
    fname = os.path.splitext(os.path.basename(fname))[0]
    return {
        'donor': fname.split('_')[0],
        'sid': fname.split('_')[1]
    }
run_canvas('ALZ', alz_filename_parser)

In [ ]:
def uc_filename_parser(fname):
    pass
run_canvas('UC')